# HDD 19 — Matched Q3 Transformer failure-warning evaluation

Evaluate Notebook 18’s three frozen reconstruction scores on precisely the Notebook 16 evaluation-drive score rows and target protocol. Compare the previously frozen five baselines descriptively. All thresholds were fitted on separate calibration drives without failure labels. Q3 labels have already been inspected in this research program, so these are exploratory development results; **do not pick a winner or claim held-out test performance**. Q4 and the sealed June test stay unopened.

Run after Notebooks 16 and 18 complete. Set `COMPONENT_ROOT` if your checkout differs. This notebook reads Q3 serial/date presence for censoring; its largest step scans the archive once. It never trains or rescales a model.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, re, zipfile
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score
try:
    from IPython.display import display
except ImportError:
    display=print

COMPONENT_ROOT=Path(r'D:/SLIIT/Y4S1/RP/Research/J26-DS-342-/01_component_1_behavioral_drift_genai')
RUN_ID='single_model_q1q2_2021_v1'
AUDIT_ID='q3_multimodel_event_coverage_v1'
COMPARE_ID='q3_matched_detector_baseline_comparison_v1'
TRANS_ID='q3_five_model_transformer_scoring_v1'
EVAL_ID='q3_matched_transformer_failure_warning_v1'
AUDIT=COMPONENT_ROOT/'results'/RUN_ID/AUDIT_ID
COMPARE=COMPONENT_ROOT/'results'/RUN_ID/COMPARE_ID
TRANS=COMPONENT_ROOT/'results'/RUN_ID/TRANS_ID
OUT=COMPONENT_ROOT/'results'/RUN_ID/EVAL_ID
OUT.mkdir(parents=True,exist_ok=True)
Q3_ZIP=COMPONENT_ROOT/'data/raw/data_Q3_2021.zip'
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(4*1024*1024),b''):h.update(block)
    return h.hexdigest()
def read_json(path):
    assert path.is_file(),f'Missing prerequisite: {path}'
    return json.loads(path.read_text())
base_done=read_json(COMPARE/'matched_comparison_complete.json')
trans_done=read_json(TRANS/'transformer_scoring_complete.json')
assert base_done['analysis_complete'] and trans_done['analysis_complete']
assert not base_done['q4_outcomes_opened'] and not trans_done['future_test_opened']
assert not base_done['sealed_June_test_opened'] and not trans_done['sealed_June_test_opened']
assert not trans_done['failure_labels_used']
base_protocol=read_json(COMPARE/'comparison_protocol.json')
base_lock=read_json(COMPARE/'score_lock.json')
settings=read_json(TRANS/'settings.json')
audit=read_json(AUDIT/'multimodel_coverage_complete.json')
assert sha256(COMPARE/'comparison_protocol.json')==base_done['protocol_sha256']==base_lock['protocol_sha256']
assert sha256(COMPARE/'score_lock.json')==base_done['score_lock_sha256']
assert sha256(TRANS/'settings.json')==trans_done['settings_sha256']
assert sha256(TRANS/'evaluation_flag_manifest.json')==trans_done['evaluation_flag_manifest_sha256']
assert sha256(TRANS/'calibrated_thresholds.json')==trans_done['thresholds_sha256']
assert sha256(Q3_ZIP)==base_protocol['q3_zip_sha256']==settings['q3_zip_sha256']
reservation_path=COMPONENT_ROOT/audit['split_path']
assert sha256(reservation_path)==base_protocol['reservation_sha256']==settings['reservation_sha256']
reservation=read_json(reservation_path)
models=list(reservation['models']);assert len(models)==5
assert base_protocol['endpoint_metric_dates']==['2021-09-01','2021-09-30']
assert base_protocol['event_dates']==['2021-09-01','2021-09-30']
assert base_protocol['horizons_days']==[7,14,30]
assert settings['detectors']==['temp_error','sector_error','joint_error']
assert settings['calibration_dates']==['2021-08-16','2021-08-31']
assert settings['quantile']==.998 and settings['labels_used'] is False
assert base_protocol['winner_selection_enabled'] is False
manifest=read_json(TRANS/'evaluation_flag_manifest.json')
assert manifest['failure_labels_used'] is False
assert sha256(TRANS/'raw_score_manifest.json')==manifest['raw_score_manifest_sha256']==trans_done['raw_score_manifest_sha256']
assert sha256(TRANS/'calibrated_thresholds.json')==manifest['thresholds_sha256']
thresholds=read_json(TRANS/'calibrated_thresholds.json')
assert set(thresholds)==set(models)
for model in models:assert set(thresholds[model])==set(settings['detectors'])
# Follow the drive-role assignment frozen in Notebook 15, as reused in Notebooks 16 and 18.
pilot_settings=read_json(COMPONENT_ROOT/'results'/RUN_ID/'q3_five_model_isolation_forest_v1'/'settings.json')
SALT=pilot_settings['salt']
def group_for(serial):
    x=int(hashlib.sha256((SALT+'|'+str(serial)).encode()).hexdigest()[:16],16)/16**16
    return 'fit' if x<.60 else ('calibration' if x<.80 else 'evaluation')
role={str(s):group_for(s) for _,g in reservation['models'].items() for s in g['development']}
model_for={str(s):m for m,g in reservation['models'].items() for s in g['development']}
reserved={str(s) for g in reservation['models'].values() for s in g['future_test']}
assert set(role).isdisjoint(reserved)
DATES=pd.date_range('2021-07-01','2021-09-30')
DAY={str(x.date()):i for i,x in enumerate(DATES)}
keys=['serial_number','model','end_date']
score_path=COMPARE/'matched_scores.parquet'
assert sha256(score_path)==base_lock['matched_scores_sha256']
protocol={'notebook16_protocol_sha256':sha256(COMPARE/'comparison_protocol.json'),
          'notebook16_score_lock_sha256':sha256(COMPARE/'score_lock.json'),
          'notebook16_metrics_sha256':sha256(COMPARE/'matched_method_metrics.csv'),
          'notebook18_settings_sha256':sha256(TRANS/'settings.json'),
          'notebook18_flag_manifest_sha256':sha256(TRANS/'evaluation_flag_manifest.json'),
          'notebook18_threshold_sha256':sha256(TRANS/'calibrated_thresholds.json'),
          'q3_zip_sha256':sha256(Q3_ZIP),'reservation_sha256':sha256(reservation_path),
          'matched_population':'Notebook 16 Aug-Sep evaluation-drive score keys, exact one-to-one join',
          'metric_dates':['2021-09-01','2021-09-30'],'horizons_days':[7,14,30],
          'targets':'Notebook 16 observed-follow-up and first-failure rules',
          'event_rule':'Notebook 16 all h pre-failure scored endpoints present',
          'transformer_methods':settings['detectors'],'winner_selection_enabled':False,
          'q4_opened':False,'sealed_June_test_opened':False}
protocol_path=OUT/'evaluation_protocol.json'
if protocol_path.exists():assert read_json(protocol_path)==protocol,'Protocol changed; use a new EVAL_ID.'
else:protocol_path.write_text(json.dumps(protocol,indent=2))
print('Verified both prerequisite completions, score manifests, Q3 archive and split.')

Verified both prerequisite completions, score manifests, Q3 archive and split.


## 1. Lock exact score-row alignment before reading failure labels

Load the frozen matched rows from Notebook 16 and all 61 daily Notebook 18 flag partitions. Enforce equality of the full `(serial_number, model, end_date)` population; a missing Transformer window is a hard error, never silently dropped. Verify each flag is the strict frozen score/threshold decision. Save a hash of the joined, label-free scores.

In [2]:
paired=pd.read_parquet(score_path,columns=keys)
for field in ['serial_number','model','end_date']:paired[field]=paired[field].astype(str)
assert len(paired)==base_lock['rows'] and not paired.duplicated(keys).any()
assert set(paired.serial_number).isdisjoint(reserved)
assert all(role.get(s)=='evaluation' and model_for.get(s)==m for s,m in paired[['serial_number','model']].drop_duplicates().itertuples(index=False,name=None))
parts=[]
expected_dates=[d for d in DAY if '2021-08-01'<=d<='2021-09-30']
assert set(manifest['evaluation_flag_sha256'])=={f'{d}.parquet' for d in expected_dates}
for date in expected_dates:
    path=TRANS/'evaluation_flags'/f'{date}.parquet'
    assert sha256(path)==manifest['evaluation_flag_sha256'][path.name]
    a=pd.read_parquet(path)
    for field in keys:a[field]=a[field].astype(str)
    assert a.end_date.eq(date).all() and a.role.eq('evaluation').all()
    assert not a.duplicated(keys).any()
    assert a.model.isin(models).all()
    for name in settings['detectors']:
        s=a[name].to_numpy(dtype=float)
        t=a[f'{name}_threshold'].to_numpy(dtype=float)
        assert np.isfinite(s).all() and np.isfinite(t).all()
        np.testing.assert_allclose(t,a.model.map(lambda m:thresholds[m][name]).to_numpy(dtype=float),rtol=0,atol=0)
        np.testing.assert_array_equal(a[f'{name}_flag'].to_numpy(dtype=bool),s>t)
    parts.append(a[keys+[field for name in settings['detectors'] for field in (name,f'{name}_flag')]])
t=pd.concat(parts,ignore_index=True)
assert not t.duplicated(keys).any()
matched=paired.merge(t,on=keys,how='outer',validate='one_to_one',indicator=True)
assert len(matched)==len(paired)==len(t) and matched._merge.eq('both').all(),matched._merge.value_counts().to_dict()
matched.drop(columns='_merge',inplace=True)
assert all(model_for.get(s)==m for s,m in matched[['serial_number','model']].drop_duplicates().itertuples(index=False,name=None))
matched.sort_values(keys,inplace=True,kind='stable');matched.reset_index(drop=True,inplace=True)
joined_path=OUT/'matched_transformer_scores.parquet'
if joined_path.exists():pd.testing.assert_frame_equal(pd.read_parquet(joined_path),matched)
else:matched.to_parquet(joined_path,index=False)
score_lock={'protocol_sha256':sha256(protocol_path),'matched_scores_sha256':sha256(joined_path),
            'notebook16_matched_scores_sha256':sha256(score_path),'rows':len(matched),
            'failure_labels_used_for_scores_or_thresholds':False}
lock_path=OUT/'transformer_score_lock.json'
if lock_path.exists():assert read_json(lock_path)==score_lock
else:lock_path.write_text(json.dumps(score_lock,indent=2))
print('Exactly matched and frozen score rows:',len(matched))
print('September rows:',int(matched.end_date.ge('2021-09-01').sum()))

Exactly matched and frozen score rows: 945649
September rows: 464437


## 2. Rebuild Notebook 16 targets and verify their support

Use the recorded first failure per development drive. Some evaluation-drive failures have no valid scored window; preserve them in the event denominator accounting. A negative endpoint requires every calendar day after the endpoint through the horizon to be observed in Q3. Endpoints with incomplete follow-up are censored, not negatives. The resulting counts must equal Notebook 16 exactly.

In [3]:
events_path=COMPONENT_ROOT/'data/evaluation_only'/RUN_ID/AUDIT_ID/'development_first_failure_support.csv'
events=pd.read_csv(events_path,dtype={'serial_number':str},parse_dates=['failure_date'])
events=events.loc[events.serial_number.map(lambda s:role.get(s)).eq('evaluation')].copy()
assert events.serial_number.is_unique and set(events.serial_number).isdisjoint(reserved)
assert all(model_for[s]==m for s,m in events[['serial_number','model']].itertuples(index=False,name=None))
first={s:DAY[str(pd.Timestamp(d).date())] for s,d in events[['serial_number','failure_date']].itertuples(index=False,name=None)}
unscoreable=events.loc[~events.serial_number.isin(matched.serial_number)].copy()
unscoreable['reason']='no_valid_scored_window_in_matched_Aug_Sep_tables'
unscoreable.to_csv(OUT/'recorded_events_without_scored_windows.csv',index=False)
old_unscoreable=pd.read_csv(COMPARE/'recorded_events_without_scored_windows.csv',dtype={'serial_number':str})
assert set(unscoreable.serial_number)==set(old_unscoreable.serial_number)
with zipfile.ZipFile(Q3_ZIP) as z:
    members={}
    for info in z.infolist():
        if info.is_dir() or '__MACOSX' in info.filename:continue
        match=re.fullmatch(r'(2021-\d{2}-\d{2})\.csv',Path(info.filename).name)
        if match and match.group(1) in DAY:
            assert match.group(1) not in members
            members[match.group(1)]=info.filename
assert set(members)==set(DAY)
observed={}
with zipfile.ZipFile(Q3_ZIP) as z:
    for date,member in sorted(members.items()):
        seen=set();bit=1<<DAY[date]
        with z.open(member) as f:
            for chunk in pd.read_csv(f,usecols=['serial_number'],dtype={'serial_number':str},chunksize=50000):
                serials=chunk.loc[chunk.serial_number.map(lambda s:role.get(s)).eq('evaluation'),'serial_number'].tolist()
                assert len(serials)==len(set(serials)) and seen.isdisjoint(serials)
                seen.update(serials)
                for serial in serials:observed[serial]=observed.get(serial,0)|bit

def all_days(bits,lo,hi):
    if lo<0 or hi>=len(DATES):return False
    mask=((1<<(hi-lo+1))-1)<<lo
    return bits&mask==mask
sept=matched.loc[matched.end_date.ge('2021-09-01')].copy().reset_index(drop=True)
assert sept.end_date.le('2021-09-30').all()
for h in protocol['horizons_days']:
    vals=[]
    for serial,date in sept[['serial_number','end_date']].itertuples(index=False,name=None):
        day=DAY[date];failure=first.get(serial)
        if failure is not None and failure<=day:v=np.nan
        elif failure is not None and failure-day<=h:v=1
        elif all_days(observed.get(serial,0),day+1,day+h):v=0
        else:v=np.nan
        vals.append(v)
    sept[f'target_h{h}']=vals
support=[]
for h in protocol['horizons_days']:
    for model in models:
        s=sept.loc[sept.model.eq(model),f'target_h{h}']
        support.append({'model':model,'horizon_days':h,'scoreable_endpoints':len(s),
                        'positive_endpoints':int(s.eq(1).sum()),'negative_endpoints':int(s.eq(0).sum()),
                        'censored_or_post_failure':int(s.isna().sum())})
support=pd.DataFrame(support)
prior=pd.read_csv(COMPARE/'matched_target_support.csv')
check=['model','horizon_days','scoreable_endpoints','positive_endpoints','negative_endpoints','censored_or_post_failure']
pd.testing.assert_frame_equal(support[check].sort_values(['model','horizon_days']).reset_index(drop=True),
                              prior[check].sort_values(['model','horizon_days']).reset_index(drop=True),check_dtype=False)
support.to_csv(OUT/'matched_target_support.csv',index=False)
assert support.loc[support.horizon_days.eq(30),'negative_endpoints'].eq(0).all()
print('Unscoreable recorded evaluation-drive failures:',len(unscoreable))
display(support)

Unscoreable recorded evaluation-drive failures: 11


,model,horizon_days,scoreable_endpoints,positive_endpoints,negative_endpoints,censored_or_post_failure
0,TOSHIBA MG07ACA14TA,7,154890,14,118744,36132
1,ST12000NM0008,7,91357,28,70019,21310
2,ST4000DM000,7,91041,49,69739,21253
3,ST8000NM0055,7,66722,21,51159,15542
4,HGST HMS5C4040BLE640,7,60427,0,46380,14047
5,TOSHIBA MG07ACA14TA,14,154890,28,82597,72265
6,ST12000NM0008,14,91357,53,48731,42573
7,ST4000DM000,14,91041,92,48451,42498
8,ST8000NM0055,14,66722,42,35636,31044
9,HGST HMS5C4040BLE640,14,60427,0,32317,28110


## 3. Evaluate the three frozen scores

Average precision is computed only when a model/horizon has both positive and observed negative endpoints. The alert count on all scored September endpoints is a workload measure; false alerts per 1,000 uses only observed negative endpoints. An event is eligible only if **all** preceding `h` dates have a matched score; strict event recall uses these complete event histories. Incomplete events remain in the event details.

In [4]:
rows=[];details=[]
for model in models:
    full=matched.loc[matched.model.eq(model)].copy()
    sub=sept.loc[sept.model.eq(model)].copy()
    for method in settings['detectors']:
        full['flag']=full[f'{method}_flag'].astype(bool)
        by_serial={s:g.set_index('end_date') for s,g in full.groupby('serial_number',sort=False)}
        scores=sub[method].to_numpy(dtype=float)
        flags=sub[f'{method}_flag'].to_numpy(dtype=bool)
        for h in protocol['horizons_days']:
            target=sub[f'target_h{h}'].to_numpy(dtype=float)
            keep=np.isfinite(target);y=target[keep].astype(int);sc=scores[keep];fl=flags[keep]
            pos=int((y==1).sum());neg=int((y==0).sum())
            tp=int((fl&(y==1)).sum());fp=int((fl&(y==0)).sum())
            drows=[]
            for event in events.loc[events.model.eq(model)].itertuples(index=False):
                fail=pd.Timestamp(event.failure_date)
                if not pd.Timestamp('2021-09-01')<=fail<=pd.Timestamp('2021-09-30'):continue
                d=DAY[str(fail.date())]
                required=[str(DATES[k].date()) for k in range(max(0,d-h),d)]
                g=by_serial.get(event.serial_number)
                complete=bool(g is not None and len(required)==h and all(x in g.index for x in required))
                alarms=g.loc[required,'flag'] if complete else pd.Series(dtype=bool)
                first_alert=required[np.flatnonzero(alarms.to_numpy())[0]] if complete and alarms.any() else None
                drows.append({'serial_number':event.serial_number,'model':model,'method':method,
                              'horizon_days':h,'failure_date':fail,'complete_warning_history':complete,
                              'recalled':bool(complete and alarms.any()),
                              'first_warning_lead_days':(fail-pd.Timestamp(first_alert)).days if first_alert else np.nan})
            ed=pd.DataFrame(drows,columns=['serial_number','model','method','horizon_days','failure_date','complete_warning_history','recalled','first_warning_lead_days'])
            complete_events=ed.loc[ed.complete_warning_history];recalled=complete_events.loc[complete_events.recalled]
            details.append(ed)
            rows.append({'model':model,'method':method,'horizon_days':h,
                         'scored_endpoints':len(sub),'positive_endpoints':pos,'negative_endpoints':neg,
                         'censored_or_post_failure':int((~keep).sum()),
                         'average_precision':float(average_precision_score(y,sc)) if pos and neg else np.nan,
                         'positive_prevalence':float(y.mean()) if len(y) else np.nan,
                         'threshold_precision':tp/(tp+fp) if tp+fp else np.nan,
                         'positive_endpoint_recall':tp/pos if pos else np.nan,
                         'false_alerts_per_1000_observed_negative_endpoints':1000*fp/neg if neg else np.nan,
                         'all_alerts_per_1000_scored_endpoints':1000*flags.mean() if len(flags) else np.nan,
                         'September_events':len(ed),'events_with_complete_warning_history':len(complete_events),
                         'events_recalled':len(recalled),
                         'strict_event_recall':len(recalled)/len(complete_events) if len(complete_events) else np.nan,
                         'median_first_warning_lead_days':float(recalled.first_warning_lead_days.median()) if len(recalled) else np.nan})
metrics=pd.DataFrame(rows)
event_details=pd.concat(details,ignore_index=True)
assert len(metrics)==len(models)*len(settings['detectors'])*len(protocol['horizons_days'])
assert metrics.loc[metrics.horizon_days.eq(30),'average_precision'].isna().all()
old=pd.read_csv(COMPARE/'matched_method_metrics.csv')
assert sha256(COMPARE/'matched_method_metrics.csv')==base_done['metrics_sha256']
reference=old.loc[old.method.eq('isolation_forest')]
cols=['model','horizon_days','scored_endpoints','positive_endpoints','negative_endpoints','censored_or_post_failure','September_events','events_with_complete_warning_history']
for method in settings['detectors']:
    a=metrics.loc[metrics.method.eq(method),cols].sort_values(['model','horizon_days']).reset_index(drop=True)
    b=reference[cols].sort_values(['model','horizon_days']).reset_index(drop=True)
    pd.testing.assert_frame_equal(a,b,check_dtype=False)
prior_events=pd.read_csv(COMPARE/'matched_event_details.csv',dtype={'serial_number':str},parse_dates=['failure_date'])
prior_events=prior_events.loc[prior_events.method.eq('isolation_forest'),['model','horizon_days','serial_number','failure_date','complete_warning_history']]
ours=event_details.loc[event_details.method.eq('joint_error'),['model','horizon_days','serial_number','failure_date','complete_warning_history']]
pd.testing.assert_frame_equal(ours.sort_values(['model','horizon_days','serial_number']).reset_index(drop=True),
                              prior_events.sort_values(['model','horizon_days','serial_number']).reset_index(drop=True),check_dtype=False)
metrics.to_csv(OUT/'transformer_method_metrics.csv',index=False)
event_details.to_csv(OUT/'transformer_event_details.csv',index=False)
display(metrics.loc[metrics.horizon_days.eq(14),['model','method','positive_endpoints','negative_endpoints','average_precision','events_with_complete_warning_history','strict_event_recall','false_alerts_per_1000_observed_negative_endpoints','all_alerts_per_1000_scored_endpoints']])

C:\Users\IMAKA\AppData\Local\Temp\ipykernel_20524\1420029833.py:46: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  event_details=pd.concat(details,ignore_index=True)


,model,method,positive_endpoints,negative_endpoints,average_precision,events_with_complete_warning_history,strict_event_recall,false_alerts_per_1000_observed_negative_endpoints,all_alerts_per_1000_scored_endpoints
1,TOSHIBA MG07ACA14TA,temp_error,28,82597,0.000489,2,0.000000,1.828154,7.495642
4,TOSHIBA MG07ACA14TA,sector_error,28,82597,0.000325,2,0.000000,1.731298,1.885209
7,TOSHIBA MG07ACA14TA,joint_error,28,82597,0.000485,2,0.000000,1.682870,6.753180
10,ST12000NM0008,temp_error,53,48731,0.001192,4,0.000000,0.369375,3.732609
13,ST12000NM0008,sector_error,53,48731,0.029395,4,0.500000,3.570622,3.721663
16,ST12000NM0008,joint_error,53,48731,0.005122,4,0.250000,1.067083,3.469904
19,ST4000DM000,temp_error,92,48451,0.003409,7,0.000000,2.208417,2.328621
22,ST4000DM000,sector_error,92,48451,0.043559,7,0.142857,1.630513,1.559737
25,ST4000DM000,joint_error,92,48451,0.036530,7,0.142857,1.671792,1.581705
28,ST8000NM0055,temp_error,42,35636,0.000793,4,0.000000,0.869907,9.202362


## 4. Matched descriptive comparison and closeout

Combine the five existing methods with three Transformer alternatives. The event table makes individual detections auditable. Small event counts and the prior Q3 inspection limit inference. No formal significance test or winner selection is performed. HGST may have no complete failure history. A 30-day endpoint AP or false-alert rate cannot be inferred from this September-only follow-up.

In [5]:
combined=pd.concat([old,metrics],ignore_index=True)
assert len(combined)==len(models)*8*3
combined.to_csv(OUT/'eight_method_matched_metrics.csv',index=False)
paired_events=pd.concat([pd.read_csv(COMPARE/'matched_event_details.csv',dtype={'serial_number':str},parse_dates=['failure_date']),
                         event_details],ignore_index=True)
assert not paired_events.duplicated(['model','method','horizon_days','serial_number','failure_date']).any()
paired_events.loc[paired_events.complete_warning_history].pivot_table(
    index=['model','horizon_days','serial_number','failure_date'],columns='method',values='recalled',aggfunc='first'
).reset_index().to_csv(OUT/'paired_complete_event_outcomes_eight_methods.csv',index=False)
summary=combined.loc[combined.horizon_days.eq(14),['model','method','average_precision','events_with_complete_warning_history',
    'events_recalled','strict_event_recall','false_alerts_per_1000_observed_negative_endpoints',
    'all_alerts_per_1000_scored_endpoints']].copy()
display(summary)
assert sha256(joined_path)==score_lock['matched_scores_sha256']
assert sha256(Q3_ZIP)==protocol['q3_zip_sha256']
record={'created_utc':datetime.now(timezone.utc).isoformat(),'analysis_complete':True,
        'protocol_sha256':sha256(protocol_path),'score_lock_sha256':sha256(lock_path),
        'transformer_metrics_sha256':sha256(OUT/'transformer_method_metrics.csv'),
        'combined_metrics_sha256':sha256(OUT/'eight_method_matched_metrics.csv'),
        'matched_rows':len(matched),'methods_compared':8,'winner_selected':False,
        'failure_labels_used_for_scoring_or_calibration':False,'failure_labels_used_for_evaluation':True,
        'q4_outcomes_opened':False,'sealed_June_test_opened':False,
        'limitations':['Q3 outcomes previously inspected; exploratory development comparison',
                       'small complete-event sample and model imbalance',
                       '30-day observed negative follow-up absent in September',
                       '0.998 calibration quantiles do not equalize September alert workloads',
                       'reconstruction error is not necessarily sensitive to every failure mode']}
(OUT/'transformer_failure_warning_complete.json').write_text(json.dumps(record,indent=2))
print('NOTEBOOK 19 COMPLETE — eight matched methods evaluated descriptively.')
print('Q4 and sealed June unopened. Results:',OUT)

,model,method,average_precision,events_with_complete_warning_history,events_recalled,strict_event_recall,false_alerts_per_1000_observed_negative_endpoints,all_alerts_per_1000_scored_endpoints
1,TOSHIBA MG07ACA14TA,isolation_forest,0.000229,2,0,0.000000,1.053307,1.039447
4,TOSHIBA MG07ACA14TA,sector_positive_fixed,0.000339,2,0,0.000000,7.239972,7.443993
7,TOSHIBA MG07ACA14TA,sector_delta_fixed,0.000339,2,0,0.000000,0.121070,0.142036
10,TOSHIBA MG07ACA14TA,sector_positive_calibrated,0.000339,2,0,0.000000,1.537586,1.575312
13,TOSHIBA MG07ACA14TA,sector_delta_calibrated,0.000339,2,0,0.000000,0.121070,0.142036
16,ST12000NM0008,isolation_forest,0.162146,4,2,0.500000,1.744270,2.003131
19,ST12000NM0008,sector_positive_fixed,0.041907,4,2,0.500000,13.872073,14.317458
22,ST12000NM0008,sector_delta_fixed,0.012885,4,2,0.500000,0.369375,0.317436
25,ST12000NM0008,sector_positive_calibrated,0.041907,4,1,0.250000,3.221768,3.568418
28,ST12000NM0008,sector_delta_calibrated,0.012885,4,2,0.500000,0.369375,0.317436


NOTEBOOK 19 COMPLETE — eight matched methods evaluated descriptively.
Q4 and sealed June unopened. Results: D:\SLIIT\Y4S1\RP\Research\J26-DS-342-\01_component_1_behavioral_drift_genai\results\single_model_q1q2_2021_v1\q3_matched_transformer_failure_warning_v1
